In [1]:
import torch
import torch.nn as nn
from dotenv import load_dotenv
import os
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

In [2]:
import sys
from pathlib import Path

project_root = str(Path('.').resolve().parent)

if project_root not in sys.path:
    sys.path.append(project_root)

In [8]:
from model import build_transformer
from data import create_data_loader, train_tokenizers

from config import ConfigLoader

In [4]:
load_dotenv(dotenv_path="../.env", override=True)
hf_token = os.getenv("HF_TOKEN")

In [5]:
config = ConfigLoader.from_yaml("../config/parameters.yaml")

In [6]:
trained_tokens_dict = train_tokenizers(
    dataset_path=config["dataset_path"],
    dataset_name=config["dataset_name"],
    source_language=config["source_language"],
    target_language=config["target_language"],
    source_file_path=config["source_file_path"],
    target_file_path=config["target_file_path"],
    max_seq_len=config["max_seq_len"],
    hf_token=hf_token,
)

Filtered dataset: 997,064 / 1,000,000 pairs kept (99.7%)


In [7]:
trained_tokens_dict["source_seq_len"], trained_tokens_dict["target_seq_len"]

(128, 128)

In [8]:
train_data_loader = create_data_loader(
    dataset_path=config["dataset_path"],
    dataset_name=config["dataset_name"],
    split="train",
    trained_tokens_dict=trained_tokens_dict,
    source_language=config["source_language"],
    target_language=config["target_language"],
    batch_size=config["train_batch_size"],
    shuffle=True,
    hf_token=hf_token,
)

In [9]:
val_data_loader = create_data_loader(
    dataset_path=config["dataset_path"],
    dataset_name=config["dataset_name"],
    split="validation",
    trained_tokens_dict=trained_tokens_dict,
    source_language=config["source_language"],
    target_language=config["target_language"],
    batch_size=config["val_batch_size"],
    shuffle=True,
    hf_token=hf_token,
)

In [10]:
model = build_transformer(
    source_vocab_size=trained_tokens_dict["source_vocab_size"],
    target_vocab_size=trained_tokens_dict["target_vocab_size"],
    source_seq_len=trained_tokens_dict["source_seq_len"],
    target_seq_len=trained_tokens_dict["target_seq_len"],
    d_model=config["d_model"],
    h=config["num_heads"],
    dropout=config["dropout"],
    n_encoder=config["n_encoder"],
    n_decoder=config["n_decoder"],
)

In [11]:
data = next(iter(train_data_loader))

In [12]:
data["source_sentence"]

["Oh, for you, I'd practice morning, noon, and night.",
 "I don't know who your contacts are, morgan, but you are my up-and-comer.",
 'Anyone!',
 "I don't feel anything.",
 'Ball is also afraid of Yashin.',
 "I'm sorry about your father.",
 'Goddammit.',
 'Why, no.',
 'the same circuit of everidentical dwelling units , offices , freeways , .',
 "It's beautiful.",
 'Cheers.',
 'Thank you for coming in.']

In [13]:
data["encoder_input"].shape, data["decoder_input"].shape, data["label"].shape

(torch.Size([12, 128]), torch.Size([12, 128]), torch.Size([12, 128]))

In [14]:
data["encoder_input"]

tensor([[    0, 10483,    56,  ...,     2,     2,     2],
        [    0,  1785,  9555,  ...,     2,     2,     2],
        [    0,     3,    23,  ...,     2,     2,     2],
        ...,
        [    0,  9888,   747,  ...,     2,     2,     2],
        [    0,     3,     4,  ...,     2,     2,     2],
        [    0,     3,  2776,  ...,     2,     2,     2]])

In [15]:
data["encoder_mask"].shape, data["decoder_mask"].shape

(torch.Size([12, 1, 1, 128]), torch.Size([12, 1, 128, 128]))

In [16]:
data["encoder_mask"][0,0,0,:]

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.])

In [17]:
data["decoder_mask"][0,0,:5,:5]

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

In [22]:
encoder_output = model.encode(data["encoder_input"], data["encoder_mask"])

In [23]:
encoder_output.shape

torch.Size([12, 128, 512])

In [24]:
decoder_output = model.decode(
    x=data["decoder_input"],
    encoder_output=encoded,
    encoder_mask=data["encoder_mask"],
    decoder_mask=data["decoder_mask"],
)

In [25]:
decoder_output.shape

torch.Size([12, 128, 512])

In [33]:
trained_tokens_dict.keys()

dict_keys(['source_tokenizer', 'target_tokenizer', 'source_seq_len', 'target_seq_len', 'source_vocab_size', 'target_vocab_size'])

In [1]:
import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter
import torchmetrics
from dotenv import load_dotenv
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
project_root = str(Path('.').resolve().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

from model import build_transformer
from data import create_data_loader, train_tokenizers
from config import ConfigLoader
from helpers.common import latest_weights_file_path

load_dotenv(dotenv_path="../.env", override=True)
hf_token = os.getenv("HF_TOKEN")





def prepare_model_data(
    config: dict,
    hf_token: str | None,
):
    trained_tokens_dict = train_tokenizers(
        dataset_path=config["dataset_path"],
        dataset_name=config["dataset_name"],
        source_language=config["source_language"],
        target_language=config["target_language"],
        tokenizer_dir=config["tokenizer_dir"],
        max_seq_len=config["max_seq_len"],
        hf_token=hf_token,
    )

    train_data_loader = create_data_loader(
        dataset_path=config["dataset_path"],
        dataset_name=config["dataset_name"],
        split="train",
        trained_tokens_dict=trained_tokens_dict,
        source_language=config["source_language"],
        target_language=config["target_language"],
        batch_size=config["train_batch_size"],
        shuffle=True,
        hf_token=hf_token,
    )

    val_data_loader = create_data_loader(
        dataset_path=config["dataset_path"],
        dataset_name=config["dataset_name"],
        split="validation",
        trained_tokens_dict=trained_tokens_dict,
        source_language=config["source_language"],
        target_language=config["target_language"],
        batch_size=config["val_batch_size"],
        shuffle=True,
        hf_token=hf_token,
    )

    model = build_transformer(
        source_vocab_size=trained_tokens_dict["source_vocab_size"],
        target_vocab_size=trained_tokens_dict["target_vocab_size"],
        source_seq_len=trained_tokens_dict["source_seq_len"],
        target_seq_len=trained_tokens_dict["target_seq_len"],
        d_model=config["d_model"],
        h=config["num_heads"],
        dropout=config["dropout"],
        n_encoder=config["n_encoder"],
        n_decoder=config["n_decoder"],
    )

    return model, train_data_loader, val_data_loader, trained_tokens_dict


def greedy_decoder(model, target_tokenizer, encoder_input, encoder_mask, device, max_translation_len):
    sos = target_tokenizer.token_to_id("<SOS>")
    eos = target_tokenizer.token_to_id("<EOS>")

    encoder_output = model.encode(encoder_input, encoder_mask)

    # translation_token_ids_so_far
    decoder_input = torch.empty(1, 1).fill_(sos).type_as(encoder_input).to(device) # (B=1, target_seq_len=1)

    last_word_token_id = None
    while last_word_token_id != eos and decoder_input.shape[1] < max_translation_len:
        # There is no padding mask; **padding to the encoder intput is only necessary for B > 1 since all batch entries should have same dim**
        causal_mask = torch.tril(torch.ones(decoder_input.shape[1], decoder_input.shape[1])).type_as(encoder_mask).to(device) 
        decoder_output = model.decode(decoder_input, encoder_output, encoder_mask, causal_mask)
        projection = model.project(decoder_output) # (B, target_seq_len, target_vocab_size) **target_seq_len growing here**

        last_word_logits = projection[:, -1] # equivalent of projection[:, -1, :] # (B=1, target_vocab_size)

        # This assumes token id is the same as the index in vocabulary
        # This works correctly because nn.Embedding and the projection layer are both indexed by token ID
        _, last_word_token_id = torch.max(last_word_logits, dim=-1) # get the index of the vocab with largest logit

        decoder_input = torch.cat([decoder_input, last_word_token_id.view(1, 1).type_as(decoder_input).to(device)], dim=1)

    return decoder_input.squeeze(0)



def run_validation(model, data_loader, target_tokenizer, target_seq_len, device, print_msg, writer, global_step, max_num_val_data=None):
    model.eval()

    try:
        # get the console window width
        with os.popen('stty size', 'r') as console:
            _, console_width = console.read().split()
            console_width = int(console_width)
    except:
        # If we can't get the console width, use 80 as default
        console_width = 80
    
    translation_sentence_ls, target_sentence_ls = [], []
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            encoder_input = batch["encoder_input"].to(device)
            encoder_mask = batch["encoder_mask"].to(device)
            assert encoder_input.shape[0] == 1, "Validitaion batch size must be 1"
            
            translation_token_ids = greedy_decoder(  # shape: (translation_seq_len)
                model=model,
                target_tokenizer=target_tokenizer,
                encoder_input=encoder_input,
                encoder_mask=encoder_mask,
                device=device,
                max_translation_len=target_seq_len
            )
            translation_sentence = target_tokenizer.decode(translation_token_ids.detach().cpu().numpy())
            target_sentence = batch["target_sentence"][0]
            source_sentence = batch["source_sentence"][0]
            
            translation_sentence_ls.append(translation_sentence)
            target_sentence_ls.append(target_sentence_ls)

            # Print the source, target and model output
            print_msg('-'*console_width)
            print_msg(f"{f'SOURCE: ':>12}{source_sentence}")
            print_msg(f"{f'TARGET: ':>12}{target_sentence}")
            print_msg(f"{f'TRANSLATION: ':>12}{translation_sentence}")

            if max_num_val_data is not None and i >= max_num_val_data:
                print_msg('='*console_width)
                break


    if writer:
        # Log the character error rate 
        metric = torchmetrics.CharErrorRate()
        cer = metric(translation_sentence, target_sentence)
        writer.add_scalar("validation CER", cer, global_step)
        writer.flush()

        # Log the word error rate 
        metric = torchmetrics.WordErrorRate()
        wer = metric(translation_sentence, target_sentence)
        writer.add_scalar("validation WER", wer, global_step)
        writer.flush()

        # Log the BLEU metric
        metric = torchmetrics.BLEUScore()
        bleu = metric(translation_sentence, target_sentence)
        writer.add_scalar("validation BLEU", bleu, global_step)
        writer.flush()



def train_model(
    config_file_path: str,
    hf_token: str,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    if (device == "cuda"):
        print(f"Device name: {torch.cuda.get_device_name(device.index)}")
        print(f"Device memory: {torch.cuda.get_device_properties(device.index).total_memory / 1024 ** 3} GB")
    torch.device(device)
    
    config = ConfigLoader.from_yaml(config_file_path)
    model, train_data_loader, val_data_loader, trained_tokens_dict = prepare_model_data(config=config, hf_token=hf_token)
    
    model = model.to(device)
    optimizer = torch.optim.AdamW(
        params=model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )
    loss_fn = nn.CrossEntropyLoss(
        ignore_index=trained_tokens_dict["source_tokenizer"].token_to_id('<PAD>'),
        label_smoothing=0.1,
    )

    # Make sure the paths are created
    Path(f"{config['model_weights_dir']}").mkdir(parents=True, exist_ok=True)
    Path(f"{config['tensorboard_log_dir']}").mkdir(parents=True, exist_ok=True)

    global_step = 0
    # Tensorboard
    writer = SummaryWriter(config["tensorboard_log_dir"])
    
    model_filename = latest_weights_file_path(config) if config["preload"] == "latest" else None
    initial_epoch = 0
    if model_filename:
        print(f"Preloading model {model_filename}")
        state = torch.load(model_filename)
        model.load_state_dict(state["model_state_dict"])
        initial_epoch = state["epoch"] + 1
        optimizer.load_state_dict(state["optimizer_state_dict"])
        global_step = state["global_step"]
    else:
        print("No model to preload, starting from scratch")

    for epoch in range(initial_epoch, config["num_epochs"]):
        batch_iterator = tqdm(train_data_loader, desc=f"Processing Epoch {epoch:02d}")
        model.train()
        for i, batch in enumerate(batch_iterator):

            if i > 50:
                break

            encoder_output = model.encode(
                batch["encoder_input"].to(device),
                batch["encoder_mask"].to(device),
            )
            decoder_output = model.decode(
                batch["decoder_input"].to(device),
                encoder_output,
                batch["encoder_mask"].to(device),
                batch["decoder_mask"].to(device),
            )
            projection = model.project(decoder_output)

            label = batch["label"].to(device).view(-1)
            loss = loss_fn(projection.view(-1, trained_tokens_dict["target_vocab_size"]), label)

            batch_iterator.set_postfix({"loss": f"{loss.item():6.3f}"})

            # Log the loss
            writer.add_scalar("train loss", loss.item(), global_step)
            writer.flush()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1

        # Run validation at the end of every epoch
        run_validation(
            model=model,
            data_loader=val_data_loader,
            target_tokenizer=trained_tokens_dict["target_tokenizer"],
            target_seq_len=trained_tokens_dict["target_seq_len"],
            device=device,
            print_msg=lambda msg: batch_iterator.write(msg),
            writer=writer,
            global_step=global_step,
            max_num_val_data=3,
        )

        model_file_name = f"{config['model_weights_dir']}/{config['model_basename']}_epoch_{epoch}.pt"
        torch.save(
            obj={
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                "global_step": global_step,
            },
            f=model_file_name,
        )


train_model(
    config_file_path="../config/parameters.yaml",
    hf_token=hf_token,
)

Using device: cpu
Filtered dataset: 948,795 / 1,000,000 pairs kept (94.9%)
No model to preload, starting from scratch


Processing Epoch 00:   0%|                                                                                                                 | 51/474398 [00:03<8:57:15, 14.71it/s, loss=9.630]
stty: 'standard input': Inappropriate ioctl for device


--------------------------------------------------------------------------------
    SOURCE: We're way past that.
    TARGET: اين موضوع ديگه گذشته
TRANSLATION: 
--------------------------------------------------------------------------------
    SOURCE: Hey, don't try your tricks on me.
    TARGET: بسه ! صد در صد امنه
TRANSLATION: 
--------------------------------------------------------------------------------
    SOURCE: - Okay. - I've got.
    TARGET: -باشه
TRANSLATION: 
--------------------------------------------------------------------------------
    SOURCE: What may I do for you, Ms. Groves?
    TARGET: چي کار ميتونم براتون کنم خانم گرووز؟
TRANSLATION: 


Processing Epoch 01:   0%|                                                                                                                | 51/474398 [00:04<11:21:19, 11.60it/s, loss=9.038]
stty: 'standard input': Inappropriate ioctl for device


--------------------------------------------------------------------------------
    SOURCE: * Joan Daemen, Vincent Rijmen, "The Design of Rijndael: AES – The Advanced Encryption Standard.
    TARGET: * Joan Daemen, Vincent Rijmen, "The Design of Rijndael: AES - The Advanced Encryption Standard.
TRANSLATION: 
--------------------------------------------------------------------------------
    SOURCE: Dr. Shannon, I'm sorry I'm late.
    TARGET: دکتر "شنون" ببخشيد دير کردم
TRANSLATION: 
--------------------------------------------------------------------------------
    SOURCE: And declared to them in public and in private,
    TARGET: سپس آشکارا و نهان (حقیقت توحید و ایمان را) برای آنان بیان داشتم!
TRANSLATION: 
--------------------------------------------------------------------------------
    SOURCE: So we had a propane leak.
    TARGET: . پس گاز پروپان نشت کرده بود
TRANSLATION: 


Processing Epoch 02:   0%|                                                                                                                | 51/474398 [00:04<10:38:05, 12.39it/s, loss=8.389]
stty: 'standard input': Inappropriate ioctl for device


--------------------------------------------------------------------------------
    SOURCE: There's a small favor I'd like to ask...
    TARGET: مي‌خوام يه لطفي در حقم بکني
TRANSLATION: 
--------------------------------------------------------------------------------
    SOURCE: We're in pursue.
    TARGET: داريمدنبالشونميکنيم.
TRANSLATION: 
--------------------------------------------------------------------------------
    SOURCE: - You guys forced me into that car, right?
    TARGET: بسه ، لئو
TRANSLATION: 
--------------------------------------------------------------------------------
    SOURCE: But once you're President, there's no scope of promotion at all!
    TARGET: اما زماني که تو رئيس جمهور بشي ! اصلاً هيچ جايي واسه ترفيع نيست
TRANSLATION: 
